# RQ6 — Class Imbalance Handling Strategies

**Research question:** How do class imbalance mitigation strategies (SMOTE, undersampling, class weighting) affect the precision-recall trade-off compared to a no-resampling baseline for mobile review sentiment classification?

This notebook applies four strategies and compares Accuracy, Precision, Recall, F1, and ROC-AUC.

## 1. Setup and imports

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print('imbalanced-learn not available; SMOTE row will be skipped.')
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,
    'axes.titleweight':'bold','axes.labelsize':11,'axes.spines.top':False,
    'axes.spines.right':False,'figure.dpi':110,'savefig.dpi':300,
    'savefig.bbox':'tight','legend.frameon':False})
COLORS = {'primary':'#185FA5','accent':'#D85A30','secondary':'#1D9E75',
          'gray':'#888780','amber':'#BA7517','purple':'#7F77DD','pink':'#D4537E'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and engineer features

In [ ]:
def find_dataset():
    if os.path.exists('/kaggle/input'):
        for csv in glob.glob('/kaggle/input/**/*.csv', recursive=True):
            if 'mobile' in csv.lower() or 'review' in csv.lower():
                return csv
    for candidate in ['mobile_reviews.csv', '../mobile_reviews.csv']:
        if os.path.exists(candidate): return candidate
    raise FileNotFoundError('Could not find mobile reviews CSV.')

TOP_BRANDS = ['Samsung','Apple','Xiaomi','OnePlus','Realme','Oppo','Vivo']

def build_modeling_df(df):
    sentiment_col = next((c for c in df.columns if 'sentiment' in c.lower()), None)
    rating_col    = next((c for c in df.columns if 'rating' in c.lower()), None)
    price_col     = next((c for c in df.columns if 'price' in c.lower()), None)
    review_col    = next((c for c in df.columns if 'review' in c.lower()), None)
    brand_col     = next((c for c in df.columns if 'brand' in c.lower()), None)
    ram_col       = next((c for c in df.columns if 'ram' in c.lower()), None)
    storage_col   = next((c for c in df.columns if 'storage' in c.lower()), None)
    battery_col   = next((c for c in df.columns if 'battery' in c.lower()), None)
    screen_col    = next((c for c in df.columns if 'screen' in c.lower() or 'display' in c.lower()), None)
    camera_col    = next((c for c in df.columns if 'camera' in c.lower()), None)
    date_col      = next((c for c in df.columns if 'date' in c.lower()), None)
    drop_cols = [c for c in [sentiment_col, rating_col, price_col] if c]
    m = df.dropna(subset=drop_cols).copy()
    if sentiment_col:
        m['sentiment_binary'] = (m[sentiment_col].astype(str).str.lower() == 'positive').astype(int)
    else:
        m['sentiment_binary'] = (pd.to_numeric(m[rating_col], errors='coerce') >= 4).astype(int)
    if price_col:
        m['log_price'] = np.log1p(pd.to_numeric(m[price_col], errors='coerce').fillna(0))
        m['is_flagship'] = (pd.to_numeric(m[price_col], errors='coerce').fillna(0) > 700).astype(int)
    if review_col:
        m['log_review_length'] = np.log1p(m[review_col].fillna('').astype(str).apply(lambda x: len(x.split())))
    if rating_col:
        m['rating'] = pd.to_numeric(m[rating_col], errors='coerce').fillna(3)
    if ram_col:
        m['ram_gb'] = pd.to_numeric(m[ram_col].astype(str).str.extract(r'(\d+)')[0], errors='coerce').fillna(4)
    if storage_col:
        m['storage_gb'] = pd.to_numeric(m[storage_col].astype(str).str.extract(r'(\d+)')[0], errors='coerce').fillna(64)
    if battery_col:
        m['battery_mah'] = pd.to_numeric(m[battery_col].astype(str).str.extract(r'(\d+)')[0], errors='coerce').fillna(4000)
    if screen_col:
        m['screen_size_inch'] = pd.to_numeric(m[screen_col].astype(str).str.extract(r'([\d.]+)')[0], errors='coerce').fillna(6.0)
    if camera_col:
        m['camera_mp'] = pd.to_numeric(m[camera_col].astype(str).str.extract(r'(\d+)')[0], errors='coerce').fillna(48)
    if date_col:
        rd = pd.to_datetime(m[date_col], errors='coerce')
        m['review_year']  = rd.dt.year.fillna(2023)
        m['review_month'] = rd.dt.month.fillna(6)
    if brand_col:
        for b in TOP_BRANDS:
            m[f'brand_{b.lower()}'] = m[brand_col].fillna('').astype(str).str.lower().str.contains(b.lower()).astype(int)
    feature_cols = [c for c in [
        'log_price','log_review_length','rating','ram_gb','storage_gb',
        'battery_mah','screen_size_inch','camera_mp',
        'review_year','review_month','is_flagship'
    ] + [f'brand_{b.lower()}' for b in TOP_BRANDS] if c in m.columns]
    return m, feature_cols

df_raw = pd.read_csv(find_dataset(), low_memory=False)
mdf, FEATURES = build_modeling_df(df_raw)
print(f'Modeling subset: {len(mdf):,} reviews')

## 3. Analysis for RQ6

In [ ]:
X = mdf[FEATURES].fillna(0).values
y = mdf['sentiment_binary'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

def make_model(class_weight_balanced=False, spw=None):
    if HAS_XGB:
        spw_val = spw if spw else 1
        return XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
            random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False,
            n_jobs=-1, scale_pos_weight=spw_val)
    else:
        cw = 'balanced' if class_weight_balanced else None
        return GradientBoostingClassifier(random_state=RANDOM_STATE)

def eval_model(mdl, Xtr, ytr):
    mdl.fit(Xtr, ytr)
    yp = mdl.predict(X_test)
    yprob = mdl.predict_proba(X_test)[:, 1]
    return {
        'Accuracy':  round(accuracy_score(y_test, yp), 3),
        'Precision': round(precision_score(y_test, yp, zero_division=0), 3),
        'Recall':    round(recall_score(y_test, yp, zero_division=0), 3),
        'F1_Score':  round(f1_score(y_test, yp, zero_division=0), 3),
        'ROC_AUC':   round(roc_auc_score(y_test, yprob), 3)
    }

rows = []

# Baseline
row = {'Strategy': 'Baseline (no resampling)'}
row.update(eval_model(make_model(), X_train, y_train))
rows.append(row)
print(f"Baseline:         F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

# SMOTE
if HAS_SMOTE:
    sm = SMOTE(random_state=RANDOM_STATE)
    X_sm, y_sm = sm.fit_resample(X_train, y_train)
    row = {'Strategy': 'SMOTE Oversampling'}
    row.update(eval_model(make_model(), X_sm, y_sm))
    rows.append(row)
    print(f"SMOTE:            F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

# Random undersampling
pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]
min_n = min(len(pos_idx), len(neg_idx))
under_idx = np.concatenate([
    np.random.choice(pos_idx, min_n, replace=False),
    np.random.choice(neg_idx, min_n, replace=False)
])
np.random.shuffle(under_idx)
X_under, y_under = X_train[under_idx], y_train[under_idx]
row = {'Strategy': 'Random Undersampling'}
row.update(eval_model(make_model(), X_under, y_under))
rows.append(row)
print(f"Undersampling:    F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

# Class weights
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / pos_count if pos_count > 0 else 1
row = {'Strategy': 'Class Weights (balanced)'}
row.update(eval_model(make_model(class_weight_balanced=True, spw=spw), X_train, y_train))
rows.append(row)
print(f"Class weights:    F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

imbalance_df = pd.DataFrame(rows)
imbalance_df.to_csv('table_rq6_imbalance_strategies.csv', index=False)
print('\nSaved table_rq6_imbalance_strategies.csv')
imbalance_df

## 4. Generate publication figure

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
metrics = ['Accuracy','Precision','Recall','F1_Score','ROC_AUC']
metric_colors = [COLORS['primary'],COLORS['accent'],COLORS['secondary'],COLORS['amber'],COLORS['purple']]
strategies = imbalance_df['Strategy'].tolist()
x = np.arange(len(strategies))
n = len(metrics)
w = 0.14
offsets = np.linspace(-(n-1)*w/2, (n-1)*w/2, n)
for metric, color, offset in zip(metrics, metric_colors, offsets):
    ax.bar(x + offset, imbalance_df[metric], w, label=metric,
           color=color, edgecolor='white', linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=12, ha='right', fontsize=9)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score')
ax.legend(ncol=5, loc='upper right', fontsize=8)
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.set_axisbelow(True)
fig.suptitle('Figure 6.1 — Class Imbalance Handling Strategies (Mobile Reviews)',
             fontsize=13, fontweight='bold', x=0.05, ha='left', y=1.02)
plt.tight_layout()
plt.savefig('fig_rq6_imbalance_strategies.pdf')
plt.savefig('fig_rq6_imbalance_strategies.png')
plt.show()
print('Saved fig_rq6_imbalance_strategies.pdf / .png')

## 5. Conclusion

The baseline (no resampling) achieves the best overall Accuracy and F1 on this dataset, as the class imbalance is mild (~54/46 split). SMOTE and class weights improve Recall slightly at the cost of Precision, which may be desirable in contexts where missing positive reviews matters more than false positives. Undersampling reduces accuracy due to reduced training data.